# Processamento dos artigos STIL 2021
Este notebook extrai dados de 30 artigos em português brasileiro do STIL 2021.

In [ ]:
import sys
import os
import json
from pathlib import Path

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../src'))

from extractor import list_docs, extract_text, clean_text, build_doc_url, extract_title, extract_abstract, extract_keywords
from processor import NLPProcessor

print("Bibliotecas importadas com sucesso!")

In [ ]:
# Configurar caminhos
INPUT_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
OUTPUT_DIR = "../output"

# Criar pastas(se não existirem)
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Os artigos selecionados serão lidos de: {INPUT_DIR}")
print(f"Os arquivos no template JSONs processados serão salvos em: {PROCESSED_DIR}")
print(f"A saída será salva em: {OUTPUT_DIR}")

# Verificar a quantidade de artigos
docs = list_docs(INPUT_DIR)
print(f"\n Encontrados {len(docs)} artigos.")

if len(docs) == 0:
    print("Nenhum artigo encontrado! Coloque os 30 artigos na pasta data/raw/")

In [ ]:
# Inicializar o processador
print("Inicializando o processador Stanza...")
processor = NLPProcessor()
print("Processador disponível.")

In [ ]:
# Lista para armazenar todos os dados
full_dataset = []

# Processar cada artigo
for index, path in enumerate(docs, 1):
    print(f"\n{'='*60}")
    print(f"Processando artigo {index}/{len(docs)}: {os.path.basename(path)}")
    print(f"{'='*60}")
    
    # Extrair texto do artigo
    print("Extraindo texto do artigo...")
    text = extract_text(path)
    text = clean_text(text)
    
    print(f"Texto extraído: {len(text)} caracteres")
    
    # Processar com Stanza
    print("Processando com Stanza (tokenização, POS, lemas)...")
    result = processor.process(text)
    
    print(f"Resultados: {len(result['tokens'])} tokens, {len(result['sentences'])} sentenças")
    
    # Calcular estatísticas
    print("Calculando estatísticas...")
    stats = processor.calculate_statistics(
        result['tokens'],
        result['pos_tags'],
        result['lemmas'],
        result['sentences']
    )
    
    print(f"Estatísticas: {stats['total_verbs']} verbos, {stats['total_nouns']} substantivos, {stats['total_adjectives']} adjetivos")

    title = extract_title(path)
    print(f"Título extraído: {title}")

    abstract = extract_abstract(path)
    print(f"Resumo extraído: {abstract}")

    keywords = extract_keywords(path)
    print(f"Palavras-chaves extraída: {keywords}")
    
    filename = os.path.basename(path)

    # Criar JSON do artigo
    json_template = {
        "titulo": title,
        "informacoes_url": build_doc_url(filename),
        "idioma": "Português",
        "storage_key": f"files/{filename}",
        "autores": [],
        "data_publicacao": "",
        "resumo": abstract,
        "keywords": keywords,
        "referencias": [],
        "artigo_completo": text,
        "artigo_tokenizado": result['tokens'],
        "pos_tagger": result['pos_tags'],
        "lema": result['lemmas']
    }
    
    full_dataset.append(json_template)
    
    # Salvar JSON no template para cada artigo 
    json_name = f"artigo_{index:02d}.json"
    json_path = os.path.join(PROCESSED_DIR, json_name)
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_template, f, ensure_ascii=False, indent=2)
    
    print(f"Artigo {json_template} salvo em: {json_path}")

    print("Top 10 palavras (com frequência):")
    
    top10 = stats['top10_list'][:10]
	
    for i, (word, frequency) in enumerate(top10, 1):
        print(f"{i:2d}. '{word}' - {frequency} ocorrências")

print(f"\n{'='*60}")
print(f"PROCESSAMENTO CONCLUÍDO. \n {len(full_dataset)} artigos processados com sucesso.")
print(f"{'='*60}")

In [ ]:
# Salvar dataset completo
dataset_path = os.path.join(OUTPUT_DIR, "full_dataset.json")

with open(dataset_path, 'w', encoding='utf-8') as f:
    json.dump(full_dataset, f, ensure_ascii=False, indent=2)

print(f"Dataset completo salvo em: {dataset_path}")
print(f"Total de artigos no dataset: {len(full_dataset)}")

In [ ]:
import pandas as pd
from collections import Counter

# Guardar estatísticas de cada artigo
stats_list = []

for doc in full_dataset:
    tokens = doc['artigo_tokenizado']
    pos_tags = doc['pos_tagger']
    
    # Contar classes gramaticais
    pos_counts = Counter(pos_tags)
    
    stats_list.append({
        'titulo': doc['titulo'],
        'total_tokens': len(tokens),
        'total_types': len(set(tokens)),
        'total_verbs': pos_counts.get('VERB', 0),
        'total_nouns': pos_counts.get('NOUN', 0) + pos_counts.get('PROPN', 0),
        'total_adjectives': pos_counts.get('ADJ', 0),
        'total_lemmas': len(set(doc['lema']))
    })

# Criar DataFrame
df_stats = pd.DataFrame(stats_list)

print("=" * 70)
print("ESTATÍSTICAS DO CORPUS (STIL 2021)")
print("=" * 70)

print(f"\nSOBRE O CORPUS:")
print(f"Total de artigos processados: {len(full_dataset)}")
print(f"Total de Tokens: {df_stats['total_tokens'].sum():,}")
print(f"Média de Tokens por artigo: {df_stats['total_tokens'].mean():.0f}")
print(f"Total de Types (palavras únicas): {df_stats['total_types'].sum():,}")
print(f"Média de Types por artigo: {df_stats['total_types'].mean():.0f}")

print(f"\nCLASSES GRAMATICAIS:")
print(f"Verbos: {df_stats['total_verbs'].sum():,}")
print(f"Substantivos: {df_stats['total_nouns'].sum():,}")
print(f"Adjetivos: {df_stats['total_adjectives'].sum():,}")
print(f"Lemas únicos: {df_stats['total_lemmas'].sum():,}")

# Top 10 palavras do corpus inteiro
all_words = []
for doc in full_dataset:
    all_words.extend([p.lower() for p in doc['artigo_tokenizado']])

top10_corpus = Counter(all_words).most_common(10)

print(f"\nTOP 10 PALAVRAS DO CORPUS (com stopwords):")
for i, (word, frequency) in enumerate(top10_corpus, 1):
    print(f"   {i}. '{word}' - {frequency:,} ocorrências")

# Estatísticas descritivas
print(f"\nESTATÍSTICAS DESCRITIVAS (Tokens por artigo):")
print(f"Mínimo: {df_stats['total_tokens'].min():,}")
print(f"Máximo: {df_stats['total_tokens'].max():,}")
print(f"Mediana: {df_stats['total_tokens'].median():.0f}")
print(f"Desvio Padrão: {df_stats['total_tokens'].std():.0f}")

# Exibir tabela resumo
print(f"\nTABELA RESUMO DOS ARTIGOS:")
print(df_stats.to_string(index=True, max_colwidth=30))